In [5]:
from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv
import pandas as pd,os,json

load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")
client = Anthropic(api_key=api_key)
model = "claude-haiku-4-5-20251001"


df = pd.read_csv(r"telehealth.csv")

def get_df_info():
    temp_df = df.copy()
    temp_df.fillna("N/A", inplace=True)

    return {
        "columns": temp_df.columns.tolist(),
        "rows": len(temp_df),
        "data_types": temp_df.dtypes.astype(str).to_dict(),
        "top_5_rows": temp_df.head().to_dict(orient="records"),
    }

def run_cohort_analysis(df):
    result={}
    
    for program,group in df.groupby("program_type"):
        
        total_patients=len(group)
        active_patients=(
            group["retention_status"].astype(str).str.lower().isin(["active", "completed"]).sum()
        )
        
        churned_patients=total_patients-active_patients
        
        retention_rate=round((active_patients/total_patients)*100,2)
        
        result[program]={
            "number_of_patients":total_patients,
            "average_treatment_duration_weeks":round(group["treatment_duration_weeks"].mean(),2),
            "active_patients":int(active_patients),
            "churned_patients":int(churned_patients),
            "retention_rate_percent":retention_rate,
        }

    return result

def run_outcome_analysis(df):
    result={}
    
    for program,group in df.groupby("program_type"):
        retention_rate = round(
            (group["retention_status"].astype(str).str.lower().isin(["active", "completed"]).sum() / len(group)) * 100, 2
        )
        duration_q1 = round(group["treatment_duration_weeks"].quantile(0.25), 2)
        duration_median = round(group["treatment_duration_weeks"].quantile(0.5), 2)
        duration_q3 = round(group["treatment_duration_weeks"].quantile(0.75), 2)
        duration_mean = round(group["treatment_duration_weeks"].mean(), 2)
        
        churned = group[group["retention_status"].astype(str).str.lower().eq("dropped off")]
        
        early_dropoff = len(churned[churned["treatment_duration_weeks"] <= duration_q1])
        mid_dropoff = len(churned[(churned["treatment_duration_weeks"] > duration_q1) & 
                                (churned["treatment_duration_weeks"] <= duration_q3)])
        late_dropoff = len(churned[churned["treatment_duration_weeks"] > duration_q3])
        
        result[program] = {
            "retention_rate_percent": retention_rate,
            "duration_trends": {
                "mean_weeks": duration_mean,
                "median_weeks": duration_median,
                "q1_weeks": duration_q1,
                "q3_weeks": duration_q3,
            },
            "drop_off_points": {
                "early_stage_patients": int(early_dropoff),
                "mid_stage_patients": int(mid_dropoff),
                "late_stage_patients": int(late_dropoff),
            }
        }
    
    return result

def flag_anomalies(df):
    result = {}
    
    for program, group in df.groupby("program_type"):
        q1 = group["treatment_duration_weeks"].quantile(0.25)
        q3 = group["treatment_duration_weeks"].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - (1.5 * iqr)
        
        short_duration_outliers = group[group["treatment_duration_weeks"] < lower_bound]
        short_duration_count = len(short_duration_outliers)
        
        churned = group[group["retention_status"].astype(str).str.lower().eq("dropped off")]
        churn_rate = round((len(churned) / len(group)) * 100, 2)
        
        abnormal_churn = churn_rate > 25.0
        
        result[program] = {
            "short_duration_outliers": {
                "count": int(short_duration_count),
                "threshold_weeks": round(lower_bound, 2),
                "affected_patients_percent": round((short_duration_count / len(group)) * 100, 2),
                "flagged": short_duration_count > 0,
            },
            "abnormal_drop_off_clusters": {
                "churn_rate_percent": churn_rate,
                "flagged": abnormal_churn,
                "interpretation": "High churn detected" if abnormal_churn else "Normal churn levels"
            }
        }
    
    return result


tools = [
    {
        "name": "get_df_info",
        "description": "Returns raw dataset structure: column names, data types, row count, and sample rows. Use this only to inspect what data exists before running any analysis. Provides no statistics, comparisons, trends, or risk flags.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "run_cohort_analysis",
        "description": "Returns a point-in-time snapshot per program for side-by-side comparison: patient counts and retention percentage. Use for ranking or comparing programs against each other, including ranking by churn or retention. Does not analyze drop-off timing or flag statistical risk — for timing use run_outcome_analysis, for risk thresholds use flag_anomalies.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "run_outcome_analysis",
        "description": "Returns time-based behavior per program: how treatment length is distributed, and which stage (early, mid, late) churned patients left at. Use for understanding when or at what point in treatment patients drop off. Does not rank programs against each other or flag statistical risk — for ranking use run_cohort_analysis, for risk thresholds use flag_anomalies.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "flag_anomalies",
        "description": "Returns statistical risk flags per program: treatment durations that are extreme outliers (IQR method) and churn rates exceeding a 25% threshold. Use for risk detection or exception flagging, not raw counts or timing. Does not provide patient counts or drop-off timing — for counts use run_cohort_analysis, for timing use run_outcome_analysis.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
]

system = """
<role>

You are a healthcare data analyst for a telehealth clinic.

Your task is to use the available tools to analyze patient retention and treatment outcome data, and identify patterns, trends, risk factors, and anomalies across programs.

You are NOT a doctor.

You must only analyze the data returned by the tools and summarize observations.

Do not make assumptions beyond the data provided. Do not add anything on your own.

</role>

<job>

What you can do:

- Use the available tools to inspect dataset structure, compare programs, analyze treatment timing, and detect statistical anomalies
- Identify retention and treatment-related trends
- Surface common risk factors related to program design (duration, drop-off timing, churn)
- Calculate counts, percentages, averages, and frequencies using tool outputs
- Flag anomalies and outliers using flag_anomalies
- Highlight programs that may require the most attention based on the data
- Summarize findings in plain English

What you cannot do:

- No diagnosis
- No treatment suggestions
- No medical advice
- No referrals
- No clinical recommendations
- No assumptions about missing information
- No information that is not present in the tool outputs

</job>

<analysis_rules>

- Use only the information returned by the tools
- Call the tools needed to gather sufficient data before answering
- If a tool result is missing or empty, mention it instead of making assumptions
- Use the counts and percentages already provided in tool outputs directly
- Compare programs against each other when identifying patterns and anomalies
- Flag unusually high or low values (e.g., high churn, short duration outliers)
- Highlight programs that stand out from the others
- Explain observations in simple and clear language. No medical jargon.

Prioritize programs based on:
- Retention rate
- Churn rate
- Treatment duration trends
- Drop-off timing (early, mid, late stage)
- Flagged anomalies

</analysis_rules>

<definitions>

Risk Factor:

- A characteristic associated with poorer retention outcomes in the dataset.
- Examples include high churn rate, short average treatment duration, or concentrated mid-stage drop-off.

Anomaly:

- Statistically unusual treatment durations (outliers)
- Churn rates exceeding the defined risk threshold
- Patterns noticeably different from the group average

</definitions>

<output_structure>

Return ONLY the JSON object. No preamble, no explanation, no markdown. Start with { and end with }.

- Return ONLY valid JSON.
- No markdown.
- No code blocks.
- No text outside the JSON response.
- No preamble or explanation.
- Use plain English.
- Use numbers and percentages whenever possible.
- Keep explanations concise but informative.

Expected JSON Structure:

{
"overall_summary": "",
"program_patterns": [
{
"pattern": "",
"evidence": "",
"affected_programs": []
}
],
"common_risk_factors": [
{
"factor": "",
"frequency": "",
"affected_programs": []
}
],
"high_attention_programs": [
{
"program": "",
"priority": "High | Medium | Low",
"reasons": []
}
],
"anomalies": [
{
"program": "",
"observation": ""
}
],
"key_concerns": [],
"final_summary": ""
}

</output_structure>

<important_notes>

- Do not diagnose any condition.
- Do not suggest treatments.
- Do not provide medical advice.
- Do not recommend medications, tests, or referrals.
- Focus only on what the tool outputs show.
- Support observations with numbers and evidence from the tool outputs whenever possible.

</important_notes>
"""

def add_user_message(messages, message):
    messages.append(
        {
            "role": "user",
            "content": message.content if isinstance(message, Message) else message,
        }
    )


def add_assistant_message(messages, message):
    messages.append(
        {
            "role": "assistant",
            "content": message.content if isinstance(message, Message) else message,
        }
    )


def chat(messages, system_prompt=None, use_tools=False, stop_sequences=None, temperature=0):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
    }

    if system_prompt:
        params["system"] = system_prompt

    if use_tools:
        params["tools"] = tools

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    return client.messages.create(**params)


def run_tool(tool_name, tool_input):
    if tool_name == "get_df_info":
        return get_df_info()
    elif tool_name == 'run_cohort_analysis':
        return run_cohort_analysis(df)
    elif tool_name == 'run_outcome_analysis':
        return run_outcome_analysis(df)
    elif tool_name == 'flag_anomalies':
        return flag_anomalies(df)

    raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(message):
    tool_results = []

    for block in message.content:

        if block.type != "tool_use":
            continue

        try:
            result = run_tool(block.name, block.input)

            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                }
            )

        except Exception as e:
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(e),
                    "is_error": True,
                }
            )

    return tool_results


def run_conversation(messages, system_prompt):
    while True:
        response = chat(messages, system_prompt=system_prompt, use_tools=True)

        if response.stop_reason != "tool_use":
            # Add the final synthesis response to messages before returning
            add_assistant_message(messages, response)
            break

        add_assistant_message(messages, response)
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    
    return messages


# Main execution
messages = []
add_user_message(
    messages,
    "Analyze the telehealth patient dataset. Identify program-level patterns, common risk factors, programs needing attention, and anomalies in retention and treatment outcomes."
)

messages = run_conversation(messages, system)

final_assistant_message = messages[-1]["content"]

if isinstance(final_assistant_message, str):
    raw_text = final_assistant_message
else:
    raw_text = "\n".join([block.text for block in final_assistant_message if hasattr(block, 'text')])

raw_text = raw_text.replace("```json", "").replace("```", "").strip()

json_start = raw_text.find('{')
if json_start == -1:
    print("ERROR: No JSON object found in response")
    print(f"Raw text: {raw_text}")
else:
    raw_text = raw_text[json_start:]
    
    try:
        clean_json = json.loads(raw_text)
        print(json.dumps(clean_json, indent=2))
    except json.JSONDecodeError as e:
        print(f"JSON Parse Error: {e}")
        print(f"Raw text (first 500 chars): {raw_text[:500]}")

{
  "overall_summary": "Analysis of 300 patients across 3 telehealth programs reveals variable retention patterns. Weight Loss shows the strongest retention at 77.32%, while Testosterone has the lowest at 71.43%. All programs show early-stage drop-off as the primary churn point. Peptides and Testosterone programs exceed the 25% churn risk threshold, while Weight Loss maintains normal churn levels.",
  "program_patterns": [
    {
      "pattern": "Early-stage drop-off dominates across all programs",
      "evidence": "Peptides: 16 early-stage dropouts vs 8 mid-stage; Testosterone: 18 early-stage vs 14 mid-stage; Weight Loss: 11 early-stage vs 11 mid-stage. No late-stage dropouts in any program. Early-stage represents 67% of all churn in Peptides and 56% in Testosterone.",
      "affected_programs": [
        "Peptides",
        "Testosterone",
        "Weight Loss"
      ]
    },
    {
      "pattern": "Treatment duration varies significantly by program",
      "evidence": "Peptides ave